Premier League

In [1]:
import os
import time
import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# ============================================================
# CONFIGURAÇÕES — PREMIER LEAGUE
# ============================================================
CSV_PRINCIPAL = 'Campeonato Premier League 2019 até 2024.csv'
ANO_INICIO = 2019  # Primeiro ano-início de temporada (2019-2020)
BASE_URL = "https://www.flashscore.com.br/futebol/inglaterra/campeonato-ingles-{inicio}-{fim}/resultados/"
URL_CALENDARIO = "https://www.flashscore.com.br/futebol/inglaterra/campeonato-ingles/calendario/" # <-- NOVA URL PARA A PL

def gerar_seasons_list():
    """
    Gera automaticamente a lista de temporadas da Premier League.
    A PL usa formato split-year (ex: 2025-2026).
    Se estamos em 2026, a temporada corrente começou em 2025 → 2025-2026.
    Gera de ANO_INICIO até a temporada corrente.
    """
    ano_atual = datetime.datetime.now().year
    # A temporada corrente começa no ano anterior (ago) e termina no ano atual (mai)
    # Ex: em março de 2026, a temporada é 2025-2026
    ano_inicio_temporada_atual = ano_atual - 1

    seasons = []
    for inicio in range(ano_inicio_temporada_atual, ANO_INICIO - 1, -1):
        fim = inicio + 1
        url = BASE_URL.format(inicio=inicio, fim=fim)
        label = f"{inicio}-{fim}"  # Ex: "2025-2026"
        seasons.append((label, url))
    return seasons

def carregar_ids_existentes(csv_path):
    """
    Lê o CSV principal e retorna um set com todos os ID_Jogo já coletados
    e o DataFrame existente.
    """
    if os.path.exists(csv_path):
        df_existente = pd.read_csv(csv_path, dtype={'ID_Jogo': str})
        ids_existentes = set(df_existente['ID_Jogo'].astype(str).tolist())
        print(f"[INFO] CSV existente carregado: {len(df_existente)} partidas, {len(ids_existentes)} IDs únicos.")
        return df_existente, ids_existentes
    else:
        print("[INFO] Nenhum CSV existente encontrado. Iniciando do zero.")
        return pd.DataFrame(), set()

def setup_driver():
    chrome_options = Options()
    # chrome_options.add_argument("--headless")  # Deixe visível para debug
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chrome_options)
    return driver

def carregar_lista_jogos(driver, url_temporada):
    print(f"--- Acessando URL: {url_temporada} ---")
    driver.get(url_temporada)
    wait = WebDriverWait(driver, 15)
    try:
        wait.until(EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))).click()
        time.sleep(1)
    except: pass

    clicks = 0
    max_clicks = 80
    while clicks < max_clicks:
        try:
            botao_mostrar_mais = driver.find_element(By.CSS_SELECTOR, 'a[data-testid="wcl-buttonLink"]')
            if "Mostrar mais jogos" in botao_mostrar_mais.text:
                driver.execute_script("arguments[0].click();", botao_mostrar_mais)
                clicks += 1
                print(f" -> [Paginação] Clique {clicks}...")
                time.sleep(2.5)
            else:
                break
        except (NoSuchElementException, Exception):
            print(" -> Lista completa carregada.")
            break
    return True

def get_match_ids(driver):
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    match_ids = []
    divs = soup.find_all('div', class_='event__match')
    if not divs:
        divs = soup.find_all('div', title="Clique para detalhes do jogo!")
    for div in divs:
        full_id = div.get('id')
        if full_id and full_id.startswith('g_1_'):
            match_ids.append(full_id.replace('g_1_', ''))
    unique_ids = list(dict.fromkeys(match_ids))
    print(f"Total de jogos encontrados nesta página: {len(unique_ids)}")
    return unique_ids

def scrape_match_data(driver, match_id, temporada_label):
    url_sumario = f"https://www.flashscore.com.br/jogo/{match_id}/"
    try:
        driver.get(url_sumario)
        wait = WebDriverWait(driver, 10)
        wait.until(EC.presence_of_element_located((By.ID, "detail")))
        time.sleep(1)
        soup_sumario = BeautifulSoup(driver.page_source, 'html.parser')

        # RODADA
        rodada_info = "N/A"
        try:
            header_els = soup_sumario.find_all(attrs={"data-testid": "wcl-scores-overline-03"})
            for el in header_els:
                header_txt = el.text.strip().upper()
                if "RODADA" in header_txt:
                    rodada_info = header_txt.split("RODADA")[-1].strip()
                    break
                elif "ROUND" in header_txt:
                    rodada_info = header_txt.split("ROUND")[-1].strip()
                    break
                elif "-" in header_txt and ("PREMIER" in header_txt or "CAMPEONATO INGLÊS" in header_txt or "INGLÊS" in header_txt):
                    rodada_info = header_txt.split("-")[-1].strip()
        except:
            rodada_info = "N/A"

        # DATA / HORA
        try:
            data_hora = soup_sumario.select_one(".duelParticipant__startTime").text.strip()
        except: data_hora = "N/A"

        # LOCAL E PÚBLICO
        local_info = "N/A"
        publico_info = "N/A"
        try:
            info_box = soup_sumario.find("div", {"data-testid": "wcl-summaryMatchInformation"})
            if info_box:
                rows = info_box.find_all("div", recursive=True)
                for row in rows:
                    text = row.text.strip().upper()
                    if "LOCAL" in text and len(text) < 10:
                        val = row.find_next_sibling("div")
                        if val: local_info = val.text.strip()
                    if "PÚBLICO" in text:
                        val = row.find_next_sibling("div")
                        if val: publico_info = val.text.strip()
        except: pass

        # ODDS
        odd_home = odd_draw = odd_away = "N/A"
        try:
            odds_spans = soup_sumario.find_all("span", {"data-testid": "wcl-oddsValue"})
            if len(odds_spans) >= 3:
                odd_home = odds_spans[0].text.strip()
                odd_draw = odds_spans[1].text.strip()
                odd_away = odds_spans[2].text.strip()
        except Exception: pass

        # ABA ESTATÍSTICAS
        try:
            # MUDANÇA: Wait isolado de 2 segundos para jogos futuros
            wait_stats = WebDriverWait(driver, 2)
            stats_tab = wait_stats.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(@href, '/estatisticas')]")))
            driver.execute_script("arguments[0].click();", stats_tab)
            wait_stats.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-testid='wcl-statistics']")))
            time.sleep(1.2)
        except: pass

        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # NOMES E PLACAR
        try:
            home = soup.select_one('.duelParticipant__home .participant__participantName').text.strip()
            away = soup.select_one('.duelParticipant__away .participant__participantName').text.strip()
        except: home, away = "Home", "Away"
        try:
            score_wrapper = soup.select_one(".detailScore__wrapper")
            spans = score_wrapper.find_all("span")
            goals_home = spans[0].text.strip()
            goals_away = spans[-1].text.strip()
        except: goals_home, goals_away = "0", "0"

        print(f" [{temporada_label}] RD:{rodada_info} | {home} {goals_home}-{goals_away} {away}")

        # ESTATÍSTICAS
        stats_dict = {}
        rows = soup.find_all('div', {'data-testid': 'wcl-statistics'})
        for row in rows:
            try:
                cat_div = row.find('div', {'data-testid': 'wcl-statistics-category'})
                if not cat_div: continue
                stat_name = cat_div.text.strip()
                val_divs = row.find_all('div', {'data-testid': 'wcl-statistics-value'})
                if len(val_divs) == 2:
                    stats_dict[stat_name] = (val_divs[0].text.strip(), val_divs[1].text.strip())
            except: continue

        def get(k): return stats_dict.get(k, ("0", "0"))

        return {
            'Temporada': temporada_label.split('-')[0],
            'Rodada': rodada_info,
            'ID_Jogo': match_id,
            'Data_Hora': data_hora,
            'Mandante': home,
            'Visitante': away,
            'Gols_Mandante': goals_home,
            'Gols_Visitante': goals_away,
            'Estadio_Local': local_info,
            'Publico': publico_info,
            'Odd_Mandante': odd_home,
            'Odd_Empate': odd_draw,
            'Odd_Visitante': odd_away,
            'xG_Mandante': get("Gols esperados (xG)")[0],
            'xG_Visitante': get("Gols esperados (xG)")[1],
            'Posse_Mandante': get("Posse de bola")[0],
            'Posse_Visitante': get("Posse de bola")[1],
            'Finalizacoes_Mandante': get("Total de finalizações")[0],
            'Finalizacoes_Visitante': get("Total de finalizações")[1],
            'No_Alvo_Mandante': get("Finalizações no alvo")[0],
            'No_Alvo_Visitante': get("Finalizações no alvo")[1],
            'Defesas_Goleiro_Mandante': get("Defesas do goleiro")[0],
            'Defesas_Goleiro_Visitante': get("Defesas do goleiro")[1],
            'Chances_Claras_Mandante': get("Chances claras")[0],
            'Chances_Claras_Visitante': get("Chances claras")[1],
            'Escanteios_Mandante': get("Escanteios")[0],
            'Escanteios_Visitante': get("Escanteios")[1],
            'Passes_Mandante': get("Passes")[0],
            'Passes_Visitante': get("Passes")[1],
            'Amarelos_Mandante': get("Cartões amarelos")[0],
            'Amarelos_Visitante': get("Cartões amarelos")[1],
            'Vermelhos_Mandante': get("Cartões vermelhos")[0],
            'Vermelhos_Visitante': get("Cartões vermelhos")[1],
            'Trave_Mandante': get("Bolas na trave")[0],
            'Trave_Visitante': get("Bolas na trave")[1],
            'Faltas_Mandante': get("Faltas")[0],
            'Faltas_Visitante': get("Faltas")[1]
        }

    except Exception as e:
        print(f"Erro no jogo {match_id}: {e}")
        return None

def main():
    # 1. Carregar dados já existentes
    df_existente, ids_existentes = carregar_ids_existentes(CSV_PRINCIPAL)

    # 2. Gerar lista de temporadas automaticamente (até a temporada corrente)
    seasons = gerar_seasons_list()
    print(f"[INFO] Temporadas a verificar: {[s[0] for s in seasons]}")

    driver = setup_driver()
    novos_dados = []

    try:
        # --- ETAPA 1: RASPAGEM DOS RESULTADOS PASSADOS ---
        for label, url in seasons:
            print(f"\n=== VERIFICANDO RESULTADOS TEMPORADA {label} ===")
            carregar_lista_jogos(driver, url)
            ids_ano = get_match_ids(driver)

            # Filtra apenas IDs que ainda NÃO estão no CSV
            ids_novos = [mid for mid in ids_ano if mid not in ids_existentes]

            if not ids_novos:
                print(f"[SKIP] Temporada {label}: todos os jogos já estão no CSV.")
                continue

            print(f"--> {len(ids_novos)} jogos NOVOS de {label} para coletar...")

            for i, match_id in enumerate(ids_novos):
                print(f"[{label}] {i+1}/{len(ids_novos)} ", end="")
                data = scrape_match_data(driver, match_id, label)
                if data:
                    novos_dados.append(data)
                    ids_existentes.add(match_id)
                time.sleep(1.5)

            # Checkpoint parcial após cada temporada
            if novos_dados:
                df_novos = pd.DataFrame(novos_dados)
                df_novos.to_csv(f'premier_league_checkpoint_novos_{label}.csv', index=False, encoding='utf-8-sig')

        # --- ETAPA 2: RASPAGEM DO CALENDÁRIO (PRÓXIMOS 10 JOGOS) ---
        print(f"\n=== VERIFICANDO CALENDÁRIO (Próximos 10 Jogos) ===")
        driver.get(URL_CALENDARIO)
        time.sleep(3) # Aguarda a página carregar

        ids_calendario = get_match_ids(driver)[:10]
        ids_novos_cal = [mid for mid in ids_calendario if mid not in ids_existentes]

        if not ids_novos_cal:
            print("[SKIP] Os próximos 10 jogos já estão no CSV ou não foram encontrados.")
        else:
            print(f"--> {len(ids_novos_cal)} jogos FUTUROS para coletar...")
            
            # Gera a label correta para a temporada atual na hora de raspar o calendário
            ano_atual = datetime.datetime.now().year
            label_atual = f"{ano_atual-1}-{ano_atual}"
            
            for i, match_id in enumerate(ids_novos_cal):
                print(f"[FUTURO] {i+1}/{len(ids_novos_cal)} ", end="")
                data = scrape_match_data(driver, match_id, label_atual)
                if data:
                    novos_dados.append(data)
                    ids_existentes.add(match_id)
                time.sleep(1.5)

    except Exception as e:
        print(f"Erro Fatal: {e}")

    finally:
        driver.quit()

        if novos_dados:
            df_novos = pd.DataFrame(novos_dados)
            print(f"\n[INFO] {len(df_novos)} jogos NOVOS coletados nesta execução.")

            # Concatenar novos dados com os existentes
            if not df_existente.empty:
                df_final = pd.concat([df_existente, df_novos], ignore_index=True)
            else:
                df_final = df_novos

            # Remover possíveis duplicatas por segurança
            df_final.drop_duplicates(subset='ID_Jogo', keep='last', inplace=True)

            # Ordenar por Temporada e Rodada
            df_final['Temporada'] = df_final['Temporada'].astype(str)
            df_final.sort_values(by=['Temporada', 'Rodada'], ascending=[True, True], inplace=True)

            # Garantir coluna Rodada na posição 2
            cols = list(df_final.columns)
            if 'Rodada' in cols:
                cols.insert(2, cols.pop(cols.index('Rodada')))
                df_final = df_final[cols]

            df_final.to_csv(CSV_PRINCIPAL, index=False, encoding='utf-8-sig')
            print(f"\n--- SUCESSO! CSV atualizado com {len(df_final)} partidas totais. ---")
            print(f"    (eram {len(df_existente)}, adicionadas {len(df_novos)} novas)")
            print(df_final[['Temporada', 'Rodada', 'Mandante', 'Visitante', 'Data_Hora']].tail(10))
        else:
            print("\n[INFO] Nenhum jogo novo encontrado. O CSV já está atualizado!")

if __name__ == "__main__":
    main()

[INFO] CSV existente carregado: 2583 partidas, 2583 IDs únicos.
[INFO] Temporadas a verificar: ['2025-2026', '2024-2025', '2023-2024', '2022-2023', '2021-2022', '2020-2021', '2019-2020']

=== VERIFICANDO RESULTADOS TEMPORADA 2025-2026 ===
--- Acessando URL: https://www.flashscore.com.br/futebol/inglaterra/campeonato-ingles-2025-2026/resultados/ ---
 -> [Paginação] Clique 1...
 -> [Paginação] Clique 2...
 -> [Paginação] Clique 3...
 -> Lista completa carregada.
Total de jogos encontrados nesta página: 306
--> 3 jogos NOVOS de 2025-2026 para coletar...
[2025-2026] 1/3  [2025-2026] RD:31 | Leeds 0-0 Brentford
[2025-2026] 2/3  [2025-2026] RD:31 | Everton 3-0 Chelsea
[2025-2026] 3/3  [2025-2026] RD:31 | Fulham 3-1 Burnley

=== VERIFICANDO RESULTADOS TEMPORADA 2024-2025 ===
--- Acessando URL: https://www.flashscore.com.br/futebol/inglaterra/campeonato-ingles-2024-2025/resultados/ ---
 -> [Paginação] Clique 1...
 -> [Paginação] Clique 2...
 -> [Paginação] Clique 3...
 -> Lista completa carreg